# Analisis dan Perbandingan Metrik Statistik Polutan (NO2, CO, SO2)

Notebook ini mendemonstrasikan perhitungan manual untuk metrik **Median Absolute Deviation** dan **Median Absolute Difference** menggunakan Python, lalu membandingkannya dengan hasil ekstraksi fitur *time-series* dari TSFEL (Time Series Feature Extraction Library).

## 1. Rumus yang Digunakan

### A. Median Absolute Deviation (MAD)
Metrik ini mengukur sebaran data dengan menghitung median dari nilai absolut simpangan tiap data terhadap median populasinya.
**Rumus:** 
$$MAD = median(\vert{}X_i - median(X)\vert{})$$

### B. Median Absolute Difference (MADiff)
Metrik ini mengukur fluktuasi lokal dengan menghitung median dari nilai absolut selisih antara titik data yang saling berurutan (berdekatan) dalam *time series*.
**Rumus:**
$$MADiff = median(\vert{}X_i - X_{i-1}\vert{})$$

---
## Perbandingan Perhitungan Manual vs TSFEL

Pada bagian ini, kita akan membuktikan dan membandingkan hasil perhitungan metrik statistik secara manual menggunakan `numpy` dengan hasil ekstraksi fitur otomatis dari *library* TSFEL. Dua metrik yang akan diuji adalah **Median Absolute Deviation** dan **Median Absolute Difference**.

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

# Definisi nama file yang akan diuji
# Sesuaikan path (lokasi folder) jika file berada di folder tertentu (misal: '../../data/polutan/')
files = {
    'NO2': {'raw': '../../data/polutan/NO2_after.csv', 'tsfel': '../../data/polutan/NO2_widang_TSFEL.csv'},
    'CO':  {'raw': '../../data/polutan/CO_after.csv', 'tsfel': '../../data/polutan/CO_widang_TSFEL.csv'},
    'SO2': {'raw': '../../data/polutan/SO2_after.csv', 'tsfel': '../../data/polutan/SO2_widang_TSFEL.csv'}
}

results = []

for pol, f in files.items():
    try:
        # 1. Load Data Mentah
        df_raw = pd.read_csv(f['raw'])
        signal = df_raw[pol].dropna().values
        
        # 2. Perhitungan Manual
        # Median Absolute Deviation
        median_val = np.median(signal)
        calc_mad = np.median(np.abs(signal - median_val))
        
        # Median Absolute Difference
        calc_madiff = np.median(np.abs(np.diff(signal)))
        
        # 3. Load Data Ekstraksi TSFEL
        df_tsfel = pd.read_csv(f['tsfel'])
        tsfel_mad = df_tsfel['median_abs_deviation'].iloc[0]
        tsfel_madiff = df_tsfel['median_abs_diff'].iloc[0]
        
        # 4. Menyimpan format ke list untuk DataFrame
        results.append({
            'Polutan': pol,
            'Metrik': 'Median Abs Deviation',
            'Manual Python': f"{calc_mad:.9f}",
            'Ekstraksi TSFEL': f"{tsfel_mad:.9f}"
        })
        
        results.append({
            'Polutan': '', # Dikosongkan agar tampilan tabel lebih rapi
            'Metrik': 'Median Abs Diff',
            'Manual Python': f"{calc_madiff:.9f}",
            'Ekstraksi TSFEL': f"{tsfel_madiff:.9f}"
        })
        
    except FileNotFoundError as e:
        print(f"File tidak ditemukan untuk {pol}: {e}")

# 5. Menampilkan hasil komparasi dalam bentuk tabel
df_results = pd.DataFrame(results)
display(df_results)

,Polutan,Metrik,Manual Python,Ekstraksi TSFEL
0,NO2,Median Abs Deviation,0.000007434,0.000007434
1,,Median Abs Diff,0.000003773,0.000003773
2,CO,Median Abs Deviation,0.002183394,0.002183394
3,,Median Abs Diff,0.001312008,0.001312008
4,SO2,Median Abs Deviation,0.000059796,0.000058761
5,,Median Abs Diff,0.000056815,0.000054662


4. Kesimpulan dan Analisis
Akurasi Perhitungan:
Metodologi perhitungan menggunakan fungsi dasar numpy (np.median(np.abs(signal - np.median(signal))) dan np.median(np.abs(np.diff(signal)))) terbukti memberikan hasil yang ekuivalen dengan fungsionalitas kompleks yang ditawarkan oleh library TSFEL.

Kecocokan Identik pada NO2 dan CO:
Hasil perhitungan manual untuk polutan NO2 dan CO 100% identik dengan hasil ekstraksi fitur default TSFEL hingga digit desimal yang sangat panjang (presisi floating point).

Penyimpangan Sangat Kecil pada SO2:
Pada data SO2, terdapat perbedaan yang sangat marjinal (pada kisaran ~0.000001 atau digit ke-6 di belakang koma). Perbedaan super kecil ini umumnya wajar dalam data science dan terjadi akibat perbedaan penanganan batas tipe data (float precision rounding), filter pra-pemrosesan di dalam sistem under-the-hood TSFEL, atau perbedaan handling asimtot saat windowing feature extraction diterapkan pada set data yang memiliki banyak fluktuasi mendekati angka nol.